### **Rôle principal du service :**
Ce script agit comme le moteur d'intelligence artificielle (LLM) de votre Système d'Aide à la Décision Clinique (CDSS). Son rôle est double :

- **Pédagogie clinique :** Enrichir les alertes de prescription (interactions, contre-indications, allergies, etc.) avec des explications médicales courtes et contextualisées, générées dynamiquement pour le médecin prescripteur.

- **Contrôle qualité :** Analyser et valider sémantiquement les justifications textuelles saisies par les médecins lorsqu'ils décident d'ignorer une alerte (override).



### **Étapes et mécanismes implémentés :**

##### **Étape 1 : Initialisation et gestion de la haute disponibilité (Fallback) :**

- Le service configure les modèles de langage via LangChain en établissant une hiérarchie stricte.
- Il tente d'abord de se connecter à une instance locale/privée via Ollama (Priorité 1).
- En cas d'échec ou de crash d'Ollama en cours de route, il bascule automatiquement sur l'API Google Gemini (Priorité 2 / Secours), garantissant ainsi que le service d'enrichissement reste disponible.

##### **Étape 2 : Structuration des Prompts spécialisés :**

- Définition de plusieurs gabarits de prompts (templates) assignant au LLM le rôle de "pharmacologue clinique".
- Chaque type d'alerte (Interaction, Allergie croisée, Posologie, Redondance, Insuffisance rénale) possède son propre prompt pour garantir une réponse très ciblée (2 à 3 phrases maximum, sans fioritures).

##### **Étape 3 : Extraction et formatage du contexte clinique (_extract_context) :**

- Avant d'interroger le LLM, le service dissèque les objets d'alerte bruts (titres, détails textuels) et fouille dans les données d'interaction (interactions_ctx) ou de médicaments (drugs_ctx).
- Il en extrait les variables propres au patient et au médicament (ex: DCI exactes, âge du patient, clairance de la créatinine, seuils de sécurité) pour nourrir le prompt.

##### **Étape 4 : Génération et enrichissement des alertes (generate_rag_explanation & enrich_alerts_with_rag) :**

- Le service filtre les alertes pour ne traiter que les plus graves (MAJOR et MODERATE).
- Il envoie le prompt complété au LLM actif.
- Il effectue un contrôle de cohérence de base (la taille de la réponse doit être comprise entre 20 et 800 caractères) avant d'injecter silencieusement l'explication générée directement dans l'objet de l'alerte.

##### **Étape 5 : Validation sémantique des forçages/overrides (validate_override_justification) :**

- Lorsqu'un médecin maintient une prescription dangereuse, il doit se justifier. Cette fonction passe le texte saisi au LLM pour évaluer s'il s'agit d'une vraie raison médicale (ex: "Bénéfice/risque favorable...") ou de "bruit" (ex: "ok", "vu", "je sais").
- Si le texte fait moins de 10 caractères, il est automatiquement rejeté sans même solliciter le LLM, économisant ainsi des ressources.
- Le LLM renvoie un JSON strict contenant un booléen de validation et un court feedback.

##### **Étape 6 : Exposition de l'état du service (get_ai_status) :**

- Une fonction utilitaire expose l'état en temps réel des connexions LLM (provider actif, modèles configurés), ce qui est typiquement utilisé pour les routes de health check de l'API.